In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
files = glob('multilingual-tts/data/*.parquet')
files

['multilingual-tts/data/train-00002-of-00004-4a6c4794806bf189.parquet',
 'multilingual-tts/data/train-00003-of-00004-f0d3b54d8a99a1ff.parquet',
 'multilingual-tts/data/train-00000-of-00004-c699f04e5a4da714.parquet',
 'multilingual-tts/data/train-00001-of-00004-15827e715edb5d3c.parquet']

In [3]:
df = pd.read_parquet(files[0])
df

,text,speaker,languages,audio
0,"""The café আমি люблю, serves القهوة that transc...",Adam,Standard Arabic & English & Bengali & Russian,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
1,تعلمتُ الغيتار في مدريد لأعزف الموسيقى الأندلس...,George,Standard Arabic & Spanish,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
2,Этот ресторан (restaurant) سب سے بہترین ہے for...,Bill,Russian & French & Urdu & English,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
3,"Kita semua, انسانوں کے لئے, partilhamos um mun...",Harry,Indonesian & Urdu & Portuguese,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
4,"Москва adalah город красивый, और मैं यहाँ खुश ...",Emily,Russian & Indonesian & Urdu & Hindi,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
...,...,...,...,...
6380,"When Tom said ""我喜欢学习中文"", his Chinese friend re...",Jessie,Mandarin Chinese & English,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
6381,"Naquela gemütliche mahalle, a criança falava ""...",Ryan,Portuguese & German & Turkish,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
6382,"Der Junge sagte ""محبت"" zu seinem Freund, währe...",Liam,German & Urdu & Indonesian,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...
6383,"La forêt noire est un lieu magique, wo die Nat...",Nicole,French & German,{'bytes': b'ID3\x04\x00\x00\x00\x00\x00#TSSE\x...


In [11]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker'].iloc[i]}"
            })
        
    return data

In [12]:
# data = loop((files[:1], 0))

In [13]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 6385/6385 [10:57<00:00,  9.72it/s]


In [16]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'multilingual-tts_audio/multilingual-tts-data-train-00002-of-00004-4a6c4794806bf189_0.mp3',
 'text': '"The café আমি люблю, serves القهوة that transcends languages."',
 'speaker': 'multilingual-tts_audio_Adam'}

In [17]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'multilingual-tts')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 105.91ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  539kB /  539kB, 2.69MB/s  
Processing Files (1 / 1): 100%|██████████|  539kB /  539kB, 1.35MB/s  
New Data Upload: 100%|██████████|  539kB /  539kB, 1.35MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.35 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/2c8f5c5a703efab15e59ee883c8a82bed87c782c', commit_message='Upload dataset', commit_description='', oid='2c8f5c5a703efab15e59ee883c8a82bed87c782c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [18]:
audio_files = [d['audio_filename'] for d in data]

with open('multilingual-tts-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [20]:
folders = glob('multilingual-tts_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

multilingual-tts_audio_neucodec
multilingual-tts_audio


In [21]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('multilingual-tts_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   2%|▏         | 29.4MB / 1.68GB,   ???B/s  
Processing Files (0 / 1):  10%|▉         |  165MB / 1.68GB,  676MB/s  
Processing Files (0 / 1):  18%|█▊        |  297MB / 1.68GB,  669MB/s  
Processing Files (0 / 1):  25%|██▌       |  428MB / 1.68GB,  665MB/s  
Processing Files (0 / 1):  33%|███▎      |  560MB / 1.68GB,  664MB/s  
Processing Files (0 / 1):  40%|███▉      |  667MB / 1.68GB,  638MB/s  
Processing Files (0 / 1):  45%|████▌     |  761MB / 1.68GB,  610MB/s  
Processing Files (0 / 1):  51%|█████     |  855MB / 1.68GB,  590MB/s  
Processing Files (0 / 1):  61%|██████    | 1.03GB / 1.68GB,  626MB/s  
Processing Files (0 / 1):  70%|██████▉   | 1.17GB / 1.68GB,  635MB/s  
Processing Files (0 / 1):  72%|███████▏  | 1.22GB / 1.68GB,  593MB/s  
Processing Files (0 / 1):  79%|███████▉  | 1.33GB / 1.68GB,  591MB/s  
Processing Files (0 / 1):  84%|████████▍ | 1.42GB / 1.68GB,  578MB/s  
Processing